<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

# قياسي BootARDL — تحليل تطبيقي على بيانات حقيقية

### اختبار التكامل المشترك ARDL بالبوتستراب (اختبار الحدود) — بدون VECM

**المؤلف:** د. مروان رودان (Dr. Merwan Roudane)

**الحزمة على PyPI:** [https://pypi.org/project/qiyasi-bootardl/0.1.0/](https://pypi.org/project/qiyasi-bootardl/0.1.0/)

**المستودع على GitHub:** [https://github.com/merwanroudane/bootardlarabic](https://github.com/merwanroudane/bootardlarabic)

**الترخيص:** GPL-3.0

---

يقدّم هذا الدفتر تطبيقاً كاملاً باللغة العربية لاختبار التكامل المشترك
**ARDL بالبوتستراب** على **بيانات اقتصادية حقيقية**: سلاسل الاقتصاد الكلي
لألمانيا الغربية (Lütkepohl 2007) — الاستهلاك والدخل والاستثمار، ربعية من
1960 إلى 1982. الهدف اختبار وجود علاقة توازنية طويلة الأجل بين **الاستهلاك**
(المتغير التابع) و**الدخل** و**الاستثمار**.

</div>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 1. خلفية نظرية موجزة

**نموذج الانحدار الذاتي للإبطاءات الموزّعة (ARDL)** يسمح باختبار التكامل
المشترك دون اشتراط أن تكون كل السلاسل من الرتبة نفسها؛ يكفي أن تكون السلاسل
متكاملة من الرتبة صفر I(0) أو الرتبة الأولى I(1) (وليست I(2)). تُكتب صيغة
تصحيح الخطأ المشروطة كالتالي:

$$\Delta y_t = \alpha_0 + \rho\, y_{t-1} + \sum_{j} \theta_j\, x_{j,t-1}
 + \sum_{i} \gamma_i\, \Delta y_{t-i}
 + \sum_{j}\sum_{i} \delta_{j,i}\, \Delta x_{j,t-i} + \varepsilon_t$$

ويعتمد **اختبار الحدود** (Pesaran, Shin & Smith 2001) على ثلاث إحصائيات:

- **إحصائية F الكلية:** تختبر الفرضية الصفرية $H_0:\ \rho = \theta_1 = \dots = \theta_k = 0$
  (لا توجد علاقة مستوى طويلة الأجل).
- **إحصائية t:** تختبر $H_0:\ \rho = 0$ على معامل المتغير التابع المُبطأ.
- **إحصائية F للمتغيرات المستقلة:** تختبر $H_0:\ \theta_1 = \dots = \theta_k = 0$
  وتُستخدم لكشف **التكامل المشترك الزائف/المتدهور** (McNown, Sam & Goh 2018؛
  Sam, McNown & Goh 2019).

### لماذا البوتستراب؟

القيم الحرجة التقاربية لاختبار PSS مبنية على عيّنات لا نهائية؛ أما في العيّنات
الصغيرة فقد تكون غير دقيقة. لذا تولّد هذه الحزمة **توزيعات صفرية بالبوتستراب**
خاصة بالعيّنة محل الدراسة عبر **بوتستراب لبواقي نموذج ARDL المشروط أحادي
المعادلة مع تثبيت المتغيرات المستقلة X** — وهو تصميم **بدون VECM**؛ فالاستدلال
**مشروط بالمتغيرات المستقلة المرصودة**. القرار الأساسي يعتمد على هذه القيم
الحرجة بالبوتستراب لأنها أدقّ للعيّنة الحالية.

</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # حفظ الرسوم دون نافذة عرض
import os

import qiyasi_bootardl
from qiyasi_bootardl import اختبار_ARDL_بالبوتستراب
from qiyasi_bootardl.datasets import تحميل_بيانات_ألمانيا

# مجلد حفظ النتائج المُستخرجة
مجلد_النتائج = os.path.join("results")
os.makedirs(مجلد_النتائج, exist_ok=True)

print("إصدار الحزمة:", qiyasi_bootardl.__version__)

إصدار الحزمة: 0.1.0


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 2. البيانات الحقيقية

سلاسل الاقتصاد الكلي لألمانيا الغربية (مليار مارك ألماني)، موسمياً معدّلة،
ربعية من 1960Q1 إلى 1982Q4 (92 مشاهدة). المصدر: ملف E1 المرافق لـ
Lütkepohl (2007)، أصلها بنك ألمانيا الفيدرالي.

</div>

In [2]:
البيانات = تحميل_بيانات_ألمانيا().set_index("التاريخ")
البيانات.head(8)

,الاستهلاك,الدخل,الاستثمار
التاريخ,,,
1960-01-01,415.0,451.0,180.0
1960-04-01,421.0,465.0,179.0
1960-07-01,434.0,485.0,185.0
1960-10-01,448.0,493.0,192.0
1961-01-01,459.0,509.0,211.0
1961-04-01,458.0,520.0,202.0
1961-07-01,479.0,521.0,207.0
1961-10-01,487.0,540.0,214.0


In [3]:
# أخذ اللوغاريتم الطبيعي (لتفسير المعاملات كمرونات)
لوغ = np.log(البيانات)
لوغ.columns = ["لو_" + c for c in البيانات.columns]
لوغ.head(5)

,لو_الاستهلاك,لو_الدخل,لو_الاستثمار
التاريخ,,,
1960-01-01,6.028279,6.111467,5.192957
1960-04-01,6.042633,6.142037,5.187386
1960-07-01,6.073045,6.184149,5.220356
1960-10-01,6.104793,6.200509,5.257495
1961-01-01,6.129050,6.232448,5.351858


In [4]:
# دالة لحفظ الجداول كملفات HTML عربية أنيقة (اتجاه RTL وألوان فاتحة)
def احفظ_جدول(df, اسم, عنوان):
    sty = (
        df.style
        .format(precision=4)
        .set_caption(عنوان)
        .set_table_styles([
            {"selector": "caption",
             "props": [("caption-side", "top"), ("font-size", "15px"),
                       ("font-weight", "bold"), ("color", "#0b5394"),
                       ("padding", "8px"), ("text-align", "right")]},
            {"selector": "th",
             "props": [("background-color", "#0b5394"), ("color", "white"),
                       ("padding", "8px 12px"), ("text-align", "center"),
                       ("border", "1px solid #d6e0ea")]},
            {"selector": "td",
             "props": [("padding", "7px 12px"), ("text-align", "center"),
                       ("border", "1px solid #e3eaf2"), ("color", "#1a1a2e")]},
            {"selector": "tr:nth-child(even) td",
             "props": [("background-color", "#f3f7fb")]},
            {"selector": "table",
             "props": [("border-collapse", "collapse"), ("direction", "rtl"),
                       ("font-family", "Tahoma, Arial, sans-serif"),
                       ("margin", "8px auto"), ("box-shadow", "0 1px 4px rgba(11,83,148,.12)")]},
        ])
    )
    html = sty.to_html()
    wrapper = (
        '<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;background:#ffffff;'
        'padding:6px">' + html + "</div>"
    )
    with open(os.path.join(مجلد_النتائج, اسم + ".html"), "w", encoding="utf-8") as f:
        f.write(wrapper)
    return sty

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

### السلاسل الزمنية للمتغيرات

</div>

In [5]:
res = اختبار_ARDL_بالبوتستراب(
    لوغ,
    المتغير_التابع="لو_الاستهلاك",
    المتغيرات_المستقلة=["لو_الدخل", "لو_الاستثمار"],
    الحالة=3,
    أقصى_إبطاء=4,
    عدد_تكرارات_البوتستراب=2000,
    مستوى_القرار=0.05,
    طباعة_التقدم=False,
    عشوائية=2024,
)
# رسم السلاسل الزمنية وحفظه
fig = res.رسم_السلاسل_الزمنية(os.path.join(مجلد_النتائج, "السلاسل_الزمنية.png"))
fig

<Figure size 1210x550 with 1 Axes>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 3. تشغيل اختبار ARDL بالبوتستراب

اخترنا الحالة الثالثة (ثابت غير مقيّد بلا اتجاه زمني)، اختيار تلقائي للإبطاءات
بمعيار AIC بحدّ أقصى 4، و2000 تكرار بوتستراب. (نفّذنا الاختبار في الخلية
السابقة لتوفير البيانات للرسم.)

</div>

In [6]:
print(res.ملخص())

اختبار ARDL بالبوتستراب للتكامل المشترك
المتغير التابع: لو_الاستهلاك
المتغيرات المستقلة: لو_الدخل، لو_الاستثمار
الحالة المحددة: الحالة الثالثة: ثابت غير مقيد وبدون اتجاه زمني
عدد المشاهدات المستخدمة: 88
معيار اختيار ARDL: معيار أكايكي AIC
فترات الإبطاء المختارة: [1, 0, 3]
عدد تكرارات البوتستراب: 2000 (فعّالة: 2000)
----------------------------------------------------
إحصائيات الاختبار:
  إحصائية F الكلية = 13.977
  إحصائية t للمتغير التابع المتأخر = -6.460
  إحصائية F للمتغيرات المستقلة المتأخرة = 20.850 (غير مشروط = 13.904)
----------------------------------------------------
القيم الاحتمالية بالبوتستراب:
  F الكلية: 0.001
  t: 0.000
  F المستقلة: 0.000
----------------------------------------------------
القيم الحرجة بالبوتستراب (عند 5%):
  F الكلية: 4.023
  t: -3.037
  F المستقلة: 5.340
----------------------------------------------------
القرار النهائي (عند 5%): وجود تكامل مشترك
تتفق الإحصائيات الثلاث (F الكلية، t، F للمتغيرات المستقلة) على رفض فرضية عدم وجود تكامل مشترك. توجد علاق

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

### التفسير الاقتصادي والمنهجي

</div>

In [7]:
print(res.تفسير())

عند مستوى معنوية 5%، توجد علاقة توازنية طويلة الأجل (تكامل مشترك) بين المتغير التابع «لو_الاستهلاك» والمتغيرات المستقلة (لو_الدخل، لو_الاستثمار).
يمكن المضي قُدماً في تقدير معاملات الأجل الطويل ونموذج تصحيح الخطأ (ECM) لتقدير سرعة التعديل نحو التوازن.
ملخص الإحصائيات المرصودة: F الكلية = 13.977، t = -6.460، F للمتغيرات المستقلة = 20.850.


In [8]:
قرار = res.قرار()
print("القرار النهائي:", قرار.label_ar)
print(قرار.detail_ar)

القرار النهائي: وجود تكامل مشترك
تتفق الإحصائيات الثلاث (F الكلية، t، F للمتغيرات المستقلة) على رفض فرضية عدم وجود تكامل مشترك. توجد علاقة توازنية طويلة الأجل.


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 4. جداول النتائج (بالعربية)

كل الجداول تُعرض وتُحفظ بصيغة HTML عربية أنيقة في مجلد `results`.

### 4.1 معاملات النموذج المشروط

</div>

In [9]:
احفظ_جدول(res.جدول_المعاملات(), "جدول_المعاملات", "معاملات النموذج المشروط")

,المعامل,الخطأ_المعياري,إحصائية_t,القيمة_الاحتمالية
const,0.0424,0.0119,3.5713,0.0006
لو_الاستهلاك.l1,-0.3338,0.0517,-6.4602,0.0000
لو_الدخل.l1,0.3345,0.0523,6.4004,0.0000
لو_الاستثمار.l1,-0.0136,0.0114,-1.1914,0.2371
D_لو_الاستهلاك.l1,-0.3300,0.0790,-4.1793,0.0001
D_لو_الاستثمار.l1,0.0368,0.0202,1.8221,0.0723
D_لو_الاستثمار.l2,0.0615,0.0186,3.3105,0.0014
D_لو_الاستثمار.l3,0.0510,0.0185,2.7487,0.0074
D_لو_الدخل,0.4298,0.0713,6.0299,0.0000
D_لو_الاستثمار,0.0641,0.0181,3.5383,0.0007


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>4.2 إحصائيات الاختبار</h3></div>

In [10]:
احفظ_جدول(res.جدول_الإحصائيات(), "جدول_الإحصائيات", "إحصائيات الاختبار الثلاث")

,الإحصائية,القيمة_الاحتمالية_التقاربية
إحصائية F الكلية,13.9765,0.0000
إحصائية t للمتغير التابع المتأخر,-6.4602,0.0000
إحصائية F للمتغيرات المستقلة المتأخرة,20.8496,0.0000


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>4.3 القيم الحرجة بالبوتستراب</h3></div>

In [11]:
احفظ_جدول(res.جدول_البوتستراب(), "جدول_البوتستراب", "القيم الحرجة بالبوتستراب")

,القيمة_الحرجة_F_الكلية,القيمة_الحرجة_t,القيمة_الحرجة_F_المستقلة_مشروط,القيمة_الحرجة_F_المستقلة_غير_مشروط
10%,3.3505,-2.6611,4.3221,3.5061
5%,4.0233,-3.0365,5.3397,4.4615
1%,5.6554,-3.7422,8.0939,6.9190


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>4.4 القيم الاحتمالية بالبوتستراب</h3></div>

In [12]:
احفظ_جدول(res.جدول_القيم_الاحتمالية(), "جدول_القيم_الاحتمالية", "القيم الاحتمالية بالبوتستراب")

,القيمة_الاحتمالية_بالبوتستراب
إحصائية F الكلية,0.0005
إحصائية t للمتغير التابع المتأخر,0.0000
إحصائية F للمتغيرات المستقلة المتأخرة,0.0000


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>4.5 الحدود التقاربية (PSS 2001)</h3></div>

In [13]:
احفظ_جدول(res.جدول_حدود_PSS(), "جدول_حدود_PSS", "حدود PSS التقاربية")

,F_الحد_الأدنى_I0,F_الحد_الأعلى_I1,t_الحد_الأدنى_I0,t_الحد_الأعلى_I1
1%,4.1300,5.0000,-3.4300,-4.1000
2.5%,3.5500,4.3800,-3.1300,-3.8000
5%,3.1000,3.8700,-2.8600,-3.5300
10%,2.6300,3.3500,-2.5700,-3.2100


<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 5. الرسوم البيانية (عربية، ألوان فاتحة)

</div>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>5.1 توزيعات البوتستراب الصفرية</h3></div>

In [14]:
res.رسم_توزيعات_البوتستراب(os.path.join(مجلد_النتائج, "توزيعات_البوتستراب.png"))

<Figure size 1650x506 with 3 Axes>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>5.2 مقارنة القرار</h3></div>

In [15]:
res.رسم_القرار(os.path.join(مجلد_النتائج, "القرار.png"))

<Figure size 1045x572 with 1 Axes>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><h3>5.3 فحص التكامل الزائف/المتدهور</h3></div>

In [16]:
res.رسم_التكامل_الزائف(os.path.join(مجلد_النتائج, "التكامل_الزائف.png"))

<Figure size 825x550 with 1 Axes>

<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;line-height:1.9">

## 6. الخلاصة

تشير النتائج إلى **وجود علاقة توازنية طويلة الأجل** بين الاستهلاك والدخل
والاستثمار في الاقتصاد الألماني الغربي خلال الفترة 1960–1982؛ إذ تتجاوز
إحصائيات الاختبار الثلاث قيمها الحرجة بالبوتستراب وتكون قيمها الاحتمالية
بالبوتستراب صغيرة جداً. القرار الأساسي مبني على القيم الحرجة بالبوتستراب
الدقيقة للعيّنة، لا على الحدود التقاربية.

> **ملاحظة منهجية:** يثبّت البوتستراب المتغيرات المستقلة X، فالاستدلال مشروط
> بها (مقايضة التصميم بدون VECM). ويجب التأكد أن المتغيرات لا تتجاوز الرتبة
> I(1) قبل الاعتماد على نتائج اختبار الحدود.

---

**المؤلف:** د. مروان رودان — **PyPI:** [qiyasi-bootardl](https://pypi.org/project/qiyasi-bootardl/0.1.0/) — **GitHub:** [bootardlarabic](https://github.com/merwanroudane/bootardlarabic)

</div>